<a href="https://colab.research.google.com/github/Taheri1400/deep-learning-projects/blob/main/6_train_the_network(TensorFlow).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش شبکه‌
</font>
</h1>

In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.data import AUTOTUNE

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مجموعه‌داده
</font>
</h2>

In [3]:
!pip install importlib_resources

In [5]:
import tensorflow_datasets as tfds

# Construct a tf.data.Dataset
ds = tfds.load('mnist', split='train', shuffle_files=True)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.I3NULF_3.0.1/mnist-train.tfrecord-[0-9][0-9][0-9][0-9][0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.I3NULF_3.0.1/mnist-test.tfrecord-[0-9][0-9][0-9][0-9][0-9…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


In [6]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تبدیل برچسب
</font>
</h3>

In [7]:
train_labels = tf.one_hot(train_labels, 10)
test_labels = tf.one_hot(test_labels, 10)

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تبدیل به مجموعه‌داده‌ی تنسورفلو
</font>
</h3>

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
نرمال‌سازی و تغییر ابعاد
</font>
</h3>

In [9]:
# Reshape the images and normalize the pixel values
train_dataset = train_dataset.map(lambda image, label: (tf.reshape(tf.cast(image, tf.float32) / 255.0, [784]), label))
test_dataset = test_dataset.map(lambda image, label: (tf.reshape(tf.cast(image, tf.float32) / 255.0, [784]), label))

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
خط لوله (Pipeline)
</font>
</h3>

In [10]:
train_dataset = (train_dataset
                 .shuffle(512)
                 .cache()
                 .batch(32)
                 .prefetch(AUTOTUNE)
                )

test_dataset = (test_dataset
                 .cache()
                 .batch(32)
                 .prefetch(AUTOTUNE)
                )

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
فرآیند آموزش
</font>
</h2>

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تابع آموزش روی یک دسته
</font>
</h3>

In [11]:
def fit_one_batch(X, y, model, loss_fn, optimizer):
  with tf.GradientTape() as tape:
    y_prime = model(X)
    loss = loss_fn(y, y_prime)

  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))

  return (y_prime, loss)

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تابع آموزش روی یک دور
</font>
</h3>

In [12]:
def fit(dataset: tf.data.Dataset, model: tf.keras.Model, loss_fn: tf.keras.losses.Loss,
optimizer: tf.optimizers.Optimizer) -> None:
  loss_sum = 0
  correct_item_count = 0
  current_item_count = 0
  print_every = 100
  batch_loss = 0
  batch_index = 0

  for (X, y) in dataset:
    (y_prime, loss) = fit_one_batch(X, y, model, loss_fn, optimizer)

    y = tf.cast(y, tf.int64)
    correct_item_count += tf.reduce_sum(tf.cast(tf.math.argmax(y_prime, axis=1) == tf.math.argmax(y, axis=1), tf.int32)).numpy()

    batch_loss = loss.numpy()
    loss_sum += batch_loss
    current_item_count += len(X)
    batch_index += 1

    # If the remainder is zero, it means that 'batch_index' is a multiple of 'print_every'
    if ((batch_index) % print_every == 0):
      batch_accuracy = correct_item_count / current_item_count * print_every
      print(f'[Batch {batch_index:>3d} - {current_item_count:>5d} items] accuracy: {batch_accuracy:>0.1f}%, loss: {batch_loss:>7f}')

  batch_accuracy = correct_item_count / current_item_count * print_every
  print(f'[Batch {batch_index:>3d} - {current_item_count:>5d} items] accuracy: {batch_accuracy:>0.1f}%, loss: {batch_loss:>7f}')

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ساختار مدل
</font>
</h3>

In [13]:
class QDense(tf.Module):
    def __init__(self, in_features, out_features, activation=tf.identity, name=None):
        super().__init__(name=name)

        # Xavier/Glorot initialization for weights
        initializer = tf.initializers.GlorotUniform()
        self.w = tf.Variable(initializer(shape=[in_features, out_features]), name='w')

        self.b = tf.Variable(tf.zeros([out_features]), name='b')

        self.activation = activation

    def __call__(self, x):
        y = tf.matmul(x, self.w) + self.b
        y = self.activation(y)
        return y

In [14]:
class QSequentialModel(tf.Module):
  def __init__(self, name=None):
    super().__init__(name=name)

    self.dense_1 = QDense(in_features=784, out_features=20, activation=tf.nn.relu)
    self.dense_2 = QDense(in_features=20, out_features=10, activation=tf.nn.softmax)

  def __call__(self, x):
    x = self.dense_1(x)
    return self.dense_2(x)

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش کامل
</font>
</h3>

In [15]:
model = QSequentialModel(name="Quera")

loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.optimizers.SGD(0.1)

print('\nFitting:')
for epoch in range(10):
  print(f'\nEpoch {epoch + 1}\n-------------------------------')
  fit(train_dataset, model, loss_fn, optimizer)


Fitting:

Epoch 1
-------------------------------
[Batch 100 -  3200 items] accuracy: 67.1%, loss: 0.483077
[Batch 200 -  6400 items] accuracy: 77.1%, loss: 0.322994
[Batch 300 -  9600 items] accuracy: 80.7%, loss: 0.420176
[Batch 400 - 12800 items] accuracy: 82.9%, loss: 0.515024
[Batch 500 - 16000 items] accuracy: 83.8%, loss: 0.573551
[Batch 600 - 19200 items] accuracy: 84.9%, loss: 0.333404
[Batch 700 - 22400 items] accuracy: 85.8%, loss: 0.338321
[Batch 800 - 25600 items] accuracy: 86.3%, loss: 0.140236
[Batch 900 - 28800 items] accuracy: 86.9%, loss: 0.260461
[Batch 1000 - 32000 items] accuracy: 87.1%, loss: 0.344662
[Batch 1100 - 35200 items] accuracy: 87.5%, loss: 0.291418
[Batch 1200 - 38400 items] accuracy: 87.8%, loss: 0.475699
[Batch 1300 - 41600 items] accuracy: 88.2%, loss: 0.242030
[Batch 1400 - 44800 items] accuracy: 88.4%, loss: 0.134177
[Batch 1500 - 48000 items] accuracy: 88.6%, loss: 0.114454
[Batch 1600 - 51200 items] accuracy: 88.8%, loss: 0.224859
[Batch 1700 - 